# Network architecture hyperparameter grid search for the constrained copula Wasserstein distance

This grid search sweeps over values for ensemble size, learning rate, and MLP width/depth.

It runs `copula_constrained_wasserstein_velocity_field` once per
(copula pair) x (hyperparameter combination) and records, for each run:

| column | meaning |
|---|---|
| `distance` | the estimated constrained distance, `sqrt(loss_energy)` measured on a **common evaluation grid** (see §3) |
| `loss_con` | continuity-equation violation on that same grid — the number that says whether `distance` means anything |
| `runtime_s` | wall clock of `Workspace.train()` only (the `L_fade` solve in `__init__` is excluded) |
| `iters_run` | iterations actually executed — less than `num_iterations` when the early-stop branch fired |
| `L_fade` | the training-free upper bound, read straight off `Workspace.L_fade` — the workspace already solves it for every run, so nothing extra is computed here |
| `under_fade` | whether `distance <= L_fade + upper_bound_tol`, i.e. whether the run converged at all |

Results go to **`architecture_grid_search_results.csv`**.

In [ ]:
import itertools, json, math, time, types
from pathlib import Path

import sys
sys.path.insert(0, str(Path.cwd().parent))  # project root, so the distance_calculation_campaign imports resolve

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from distance_calculation_campaign.copula_families import load_target, MixtureCopula, KhoudrajiCopula, MirroredCopula
from distance_calculation_campaign.copula_constrained_wasserstein_velocity_field import Workspace, device

print('device:', device)
RESULTS_CSV = Path('architecture_grid_search_results.csv')  # appended to after every run, so an interrupted sweep is not lost

device: cuda:0


## 1. Copula pairs

Same four pairs as the training hyperparameter sweep, so the two sets of results can be read
side by side.

Each entry is `(name, factory)`, where `factory()` returns a fresh `(copula0, copula1)`.

In [2]:
SIGMA = 0.26

PAIRS = [
    ('gaussian0.9 -> gaussian-0.9',
     lambda: (load_target('GaussianCopula', 0.9, SIGMA),
              load_target('GaussianCopula', -0.9, SIGMA))),
    ('gaussian0.9 -> independence',
     lambda: (load_target('GaussianCopula', 0.9, SIGMA),
              load_target('IndependenceCopula', None, SIGMA))),
    ('frank25 -> clayton15',
     lambda: (load_target('FrankCopula', 25.0, SIGMA),
              load_target('ClaytonCopula', 15.0, SIGMA))),
    ('joe4 -> joe4@90',
     lambda: (load_target('JoeCopula', 4.0, SIGMA),
              MirroredCopula(load_target('JoeCopula', 4.0, SIGMA), 0))),
]

for name, factory in PAIRS:
    c0, c1 = factory()
    assert float(c0.sigma) == float(c1.sigma), f'{name}: endpoints must share sigma'
print(f'{len(PAIRS)} pairs')

4 pairs


## 2. Hyperparameter grid

`SWEEP` holds one short list per hyperparameter to vary; `BASE` holds everything held fixed.

The three swept knobs, and what each actually changes:

- **`num_models`** — the number of MLPs spaced along the time axis, for each of the three
  ensembles (`logrhonn`, `vector`, `constant`). Note it is *not* a pure capacity knob: the
  RBF time-weighting width is `exp(-(t - t_i)^2 * num_models^2)`, so raising it also makes
  each MLP more sharply time-localized. More models means more parameters *and* less
  smoothing across `t`.
- **`lr`** — Adam's learning rate for the single optimizer over all three ensembles.
- **`features`** — hidden layer sizes; the MLP builders wrap this as
  `[ndim+1] + features + [out]`.

Held fixed and *not* swept here:

- `n_point=16` and `window=10000`, the discretization this sweep is conditioned on.
  `window <= num_iterations` still has to hold, and `num_iterations >= 2*window` for the
  plateau check to fire more than once — 200000 vs 10000 leaves plenty of room.
- `jitter_grid=True`, `penalty_weight_increment=1`, `energy_plateau_rel_tol=0.005`, as in
  the other sweep.
- `penalty_weight` is the *starting* value; `Workspace.train()` raises it on plateaus, so
  `penalty_weight_final` is recorded separately below.

In [ ]:
SWEEP = {
    'num_models': [4, 6],
    'lr':         [1e-3, 1e-2],
    'features':   [[128, 128, 128], [128, 128], [256, 256, 256]],
}

# Held fixed across the whole sweep.
BASE = dict(
    n_point=16,                     
    window=10000,                   
    num_iterations=200000,
    penalty_weight=5,
    penalty_weight_increment=1,   
    con_tol=0.05,
    upper_bound_tol=0.02,
    jitter_grid=True,              
    energy_plateau_rel_tol=0.005, 
)

SEED = 0          
EVAL_N_POINT = 31  

CONFIGS = [dict(zip(SWEEP, vals)) for vals in itertools.product(*SWEEP.values())]
print(f'{len(CONFIGS)} configs x {len(PAIRS)} pairs = {len(CONFIGS)*len(PAIRS)} training runs')
for c in CONFIGS:
    print('   ', c)
print('\nestimate the total: (runs) x (seconds per run) -- budget is 5000 GPU-h/year, max 3 concurrent jobs')

12 configs x 4 pairs = 48 training runs
    {'num_models': 4, 'lr': 0.001, 'features': [128, 128, 128]}
    {'num_models': 4, 'lr': 0.001, 'features': [128, 128]}
    {'num_models': 4, 'lr': 0.001, 'features': [256, 256, 256]}
    {'num_models': 4, 'lr': 0.01, 'features': [128, 128, 128]}
    {'num_models': 4, 'lr': 0.01, 'features': [128, 128]}
    {'num_models': 4, 'lr': 0.01, 'features': [256, 256, 256]}
    {'num_models': 6, 'lr': 0.001, 'features': [128, 128, 128]}
    {'num_models': 6, 'lr': 0.001, 'features': [128, 128]}
    {'num_models': 6, 'lr': 0.001, 'features': [256, 256, 256]}
    {'num_models': 6, 'lr': 0.01, 'features': [128, 128, 128]}
    {'num_models': 6, 'lr': 0.01, 'features': [128, 128]}
    {'num_models': 6, 'lr': 0.01, 'features': [256, 256, 256]}

estimate the total: (runs) x (seconds per run) -- budget is 5000 GPU-h/year, max 3 concurrent jobs


## 3. Scoring on a common grid

`evaluate_losses` recomputes exactly the `loss_energy` / `loss_con` block of
`Workspace.train()`, but on one fixed regular `(n, n, n)` grid with no jitter, so every
config is scored identically.

In [4]:
def evaluate_losses(module, target0, target1, n_point=EVAL_N_POINT):
    """(loss_energy, loss_con) for a trained module on a fixed regular grid.

    Mirrors the loss computation in `Workspace.train()`. No `torch.no_grad()`:
    `module.d_model` goes through `torch.func.jacrev` and needs autograd live.
    """
    t = torch.linspace(0, 1, n_point).double().to(device)
    x = torch.linspace(-1, 1, n_point).double().to(device)
    state = torch.cartesian_prod(t, x, x)
    t_list, x_list = state[..., 0], state[..., 1:]

    logf0, logf1 = target0.log_density(x_list), target1.log_density(x_list)
    dx_logden0, dx_logden1 = target0.dx_logden(x_list), target1.dx_logden(x_list)

    logrhonn, vec, C = module(state)
    logrhonn, C = logrhonn.squeeze(), C.squeeze()
    dtlogrhonn, dxlogrhonn, dvec = module.d_model(state)

    logrho = (1-t_list)*logf0 + t_list*logf1 + t_list*(1-t_list)*logrhonn
    rho = torch.exp(logrho)
    dt_logrho = -logf0 + logf1 + (1-2*t_list)*logrhonn + t_list*(1-t_list)*dtlogrhonn
    dx_logrho = ((1-t_list).unsqueeze(-1)*dx_logden0 + t_list.unsqueeze(-1)*dx_logden1
                 + (t_list*(1-t_list)).unsqueeze(-1)*dxlogrhonn)

    rho_reshape = rho.reshape((n_point, n_point, n_point))
    f_reshape = rho_reshape / (rho_reshape.mean([1, 2]) * 2**2).unsqueeze(-1).unsqueeze(-1).repeat(1, n_point, n_point)
    f = f_reshape.reshape(rho.shape)

    loss_con = torch.abs((dt_logrho + (dx_logrho*vec).sum(1) + dvec + C)*f).mean() * 2**2

    f1 = f_reshape.mean(2) * 2
    f2 = f_reshape.mean(1) * 2
    f1 = f1.unsqueeze(-1).repeat(1, 1, n_point).reshape(f.unsqueeze(-1).shape)
    f2 = f2.unsqueeze(1).repeat(1, n_point, 1).reshape(f.unsqueeze(-1).shape)
    fv = f.unsqueeze(-1) * vec
    fv_1 = fv[..., 0].reshape((n_point,)*3).mean(2).unsqueeze(-1).repeat(1, 1, n_point).reshape(f.unsqueeze(-1).shape) * 2
    fv_2 = fv[..., 1].reshape((n_point,)*3).mean(1).unsqueeze(1).repeat(1, n_point, 1).reshape(f.unsqueeze(-1).shape) * 2
    w = torch.cat([f1, f2], dim=1)*vec - torch.cat([fv_1, fv_2], dim=1)
    loss_energy = ((w**2).sum(1)*f).mean() * 2**2

    return float(loss_energy.detach()), float(loss_con.detach())

## 4. One run

In [5]:
def run_one(pair_name, factory, config, seed=SEED, verbose=False):
    """Train one (pair, config) and return a flat result row."""
    torch.manual_seed(seed)
    c0, c1 = factory()
    args = types.SimpleNamespace(ndim=2, verbose=verbose, **{**BASE, **config})
    args.features = list(args.features)   # the MLP builders do [x] + args.features + [y]

    ws = Workspace(args, c0, c1)          # __init__ also solves the fade bound -> ws.L_fade
    t0 = time.perf_counter()
    ws.train()                            # Ctrl-C returns the partially-trained model
    runtime = time.perf_counter() - t0

    energy, con = evaluate_losses(ws.get_model(), ws.target0, ws.target1)
    distance = math.sqrt(max(energy, 0.0))

    # ws.L_fade is the training-free upper bound the workspace computes for every run
    # (None if that solve failed). A distance above it means the run did not converge --
    # the linear-mixture path, which needs no training at all, is already cheaper.
    under_fade = None if ws.L_fade is None else distance <= ws.L_fade + args.upper_bound_tol

    row = {
        'pair': pair_name,
        **{k: (json.dumps(v) if isinstance(v, list) else v) for k, v in config.items()},
        'distance': distance,
        'loss_con': con,
        'loss_energy': energy,
        'runtime_s': runtime,
        'iters_run': ws.iter,
        'early_stopped': ws.iter <= args.num_iterations,
        'penalty_weight_final': args.penalty_weight,
        'con_ok': con < BASE['con_tol'],
        'L_fade': ws.L_fade,
        'under_fade': under_fade,
        'seed': seed,
        'config_key': json.dumps(config, sort_keys=True, default=str),
    }
    del ws
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return row

## 5. The sweep

Appends each row to `RESULTS_CSV` as soon as it finishes and skips `(pair, config_key)`
combinations already present, so the cell can be re-run after an interruption, a kernel
restart, or a hit `walltime` and it will pick up where it stopped.

A run that raises is caught, recorded with an `error` column, and the sweep continues.

Delete `architecture_grid_search_results.csv` to start over.

In [6]:
def append_row(row, path=RESULTS_CSV):
    """Append one result row, reconciling columns with whatever is already in the file.

    A failed run carries an `error` column that successful rows do not, and a bare
    `to_csv(mode='a')` writes fields positionally -- an extra key against a header that
    lacks it silently shifts every later column. So: if the row introduces new columns,
    rewrite the file with the union; otherwise append in the file's own column order.
    """
    new = pd.DataFrame([row])
    if not path.exists():
        new.to_csv(path, index=False)
        return
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    if set(new.columns) - set(cols):
        pd.concat([pd.read_csv(path), new], ignore_index=True).to_csv(path, index=False)
    else:
        new.reindex(columns=cols).to_csv(path, mode='a', header=False, index=False)


done = set()
if RESULTS_CSV.exists():
    prev = pd.read_csv(RESULTS_CSV)
    done = set(zip(prev['pair'], prev['config_key']))
    print(f'{len(done)} runs already recorded, skipping those')

total = len(PAIRS) * len(CONFIGS)
k = 0
for (pair_name, factory), config in itertools.product(PAIRS, CONFIGS):
    k += 1
    key = (pair_name, json.dumps(config, sort_keys=True, default=str))
    if key in done:
        continue
    print(f'[{k}/{total}] {pair_name} | {config}', flush=True)
    try:
        row = run_one(pair_name, factory, config)
        print(f"    distance={row['distance']:.4f}  loss_con={row['loss_con']:.3e}  "
              f"{row['runtime_s']:.0f}s  iters={row['iters_run']}  under_fade={row['under_fade']}", flush=True)
    except Exception as e:
        # A diverging config must not take the other 47 runs down with it. Job 1678961
        # lost 15.6h of completed work this way: run 24/48 (num_models=6, lr=1e-2,
        # [256,256,256]) hit `Gradient norm is nan while loss is nan` -- the guard in
        # Workspace.train -- and the exception propagated out of the notebook.
        # Recording it as a row means the resume skips it AND that divergence shows up
        # as a result, which is exactly what a stability sweep should be measuring.
        # KeyboardInterrupt is a BaseException, so Ctrl-C still stops the sweep.
        row = {'pair': pair_name,
               **{kk: (json.dumps(v) if isinstance(v, list) else v) for kk, v in config.items()},
               'seed': SEED, 'config_key': key[1],
               'error': f'{type(e).__name__}: {e}'}
        print(f'    FAILED  {row["error"]}', flush=True)
    # after the except block, so `e` (and the traceback holding the Workspace alive) is gone
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    append_row(row)
    done.add(key)
print('done')

23 runs already recorded, skipping those
[24/48] gaussian0.9 -> independence | {'num_models': 6, 'lr': 0.01, 'features': [256, 256, 256]}


    FAILED  RuntimeError: Gradient norm is nan while loss is nan


[25/48] frank25 -> clayton15 | {'num_models': 4, 'lr': 0.001, 'features': [128, 128, 128]}


/hkfs/home/haicore/ior/fi1281/copula_fade_wasserstein_distance_script.py:914: UserWarning: fade_distance: both copulas have negligible density over 60% of the domain, so the fade path spends much of its time moving mass through near-empty regions. L is still a valid finite length, but a large value here is real -- it is what the linear-mixture path genuinely costs, not a numerical artefact.
  warnings.warn(f"fade_distance: {msg}. L is still a valid finite "


    distance=0.0321  loss_con=1.733e-02  1908s  iters=49999  under_fade=True


...

[47/48] joe4 -> joe4@90 | {'num_models': 6, 'lr': 0.01, 'features': [128, 128]}


    distance=0.4357  loss_con=3.262e-02  807s  iters=19999  under_fade=True


[48/48] joe4 -> joe4@90 | {'num_models': 6, 'lr': 0.01, 'features': [256, 256, 256]}


    distance=0.4350  loss_con=3.661e-02  1873s  iters=29999  under_fade=True


done


## 6. Results

In [7]:
df = pd.read_csv(RESULTS_CSV)
if 'error' not in df:
    df['error'] = pd.NA
failed = df[df['error'].notna()]
df_ok = df[df['error'].isna()].copy()          # only completed runs carry usable metrics
df_ok['gap_to_fade'] = df_ok['L_fade'] - df_ok['distance']   # positive = beat the training-free path

print(f'{len(df_ok)} completed, {len(failed)} diverged, {len(df)} attempted')
if len(failed):
    print('\ndiverged configs:')
    print(failed[['pair', *SWEEP, 'error']].to_string(index=False))
df_ok.sort_values(['pair', 'distance'])

47 completed, 1 diverged, 48 attempted

diverged configs:
                       pair  num_models   lr        features                                                error
gaussian0.9 -> independence           6 0.01 [256, 256, 256] RuntimeError: Gradient norm is nan while loss is nan


,pair,num_models,lr,features,distance,loss_con,loss_energy,runtime_s,iters_run,early_stopped,penalty_weight_final,con_ok,L_fade,under_fade,seed,config_key,error,gap_to_fade
26,frank25 -> clayton15,4,0.001,"[256, 256, 256]",0.031333,0.022330,0.000982,1758.892494,39999.0,True,5.0,True,0.024666,True,0,"{""features"": [256, 256, 256], ""lr"": 0.001, ""nu...",NaN,-0.006667
24,frank25 -> clayton15,4,0.001,"[128, 128, 128]",0.032128,0.017330,0.001032,1907.837847,49999.0,True,6.0,True,0.024666,True,0,"{""features"": [128, 128, 128], ""lr"": 0.001, ""nu...",NaN,-0.007462
30,frank25 -> clayton15,6,0.001,"[128, 128, 128]",0.033200,0.027749,0.001102,1660.917617,29999.0,True,5.0,True,0.024666,True,0,"{""features"": [128, 128, 128], ""lr"": 0.001, ""nu...",NaN,-0.008534
28,frank25 -> clayton15,4,0.010,"[128, 128]",0.034029,0.035596,0.001158,5868.279074,200001.0,False,19.0,True,0.024666,True,0,"{""features"": [128, 128], ""lr"": 0.01, ""num_mode...",NaN,-0.009363
32,frank25 -> clayton15,6,0.001,"[256, 256, 256]",0.034576,0.041810,0.001196,1260.950517,19999.0,True,6.0,True,0.024666,True,0,"{""features"": [256, 256, 256], ""lr"": 0.001, ""nu...",NaN,-0.009910
25,frank25 -> clayton15,4,0.001,"[128, 128]",0.036202,0.023500,0.001311,3231.639534,109999.0,True,6.0,True,0.024666,True,0,"{""features"": [128, 128], ""lr"": 0.001, ""num_mod...",NaN,-0.011536
31,frank25 -> clayton15,6,0.001,"[128, 128]",0.036448,0.034215,0.001328,1251.073289,29999.0,True,6.0,True,0.024666,True,0,"{""features"": [128, 128], ""lr"": 0.001, ""num_mod...",NaN,-0.011782
34,frank25 -> clayton15,6,0.010,"[128, 128]",0.036890,0.040458,0.001361,8126.601851,200001.0,False,23.0,True,0.024666,True,0,"{""features"": [128, 128], ""lr"": 0.01, ""num_mode...",NaN,-0.012224
33,frank25 -> clayton15,6,0.010,"[128, 128, 128]",0.039426,0.049008,0.001554,10872.607498,200001.0,False,18.0,True,0.024666,True,0,"{""features"": [128, 128, 128], ""lr"": 0.01, ""num...",NaN,-0.014760
27,frank25 -> clayton15,4,0.010,"[128, 128, 128]",0.041030,0.065338,0.001683,7628.504625,200001.0,False,17.0,False,0.024666,True,0,"{""features"": [128, 128, 128], ""lr"": 0.01, ""num...",NaN,-0.016364


In [8]:
# Best (smallest valid distance) config per pair.
# `.eq(True)` rather than `.fillna(False).astype(bool)`: once a diverged row puts a NaN in
# these columns pandas reads them back as object dtype, and `astype(bool)` on an object
# column maps the *string* 'False' to True. `.eq(True)` is correct for bool, NaN and str
# alike, and raises no downcasting FutureWarning.
valid = df_ok[df_ok['con_ok'].eq(True) & df_ok['under_fade'].eq(True)]
print('valid runs:', len(valid), 'of', len(df_ok), 'completed')
valid.loc[valid.groupby('pair')['distance'].idxmin()] if len(valid) else 'no valid runs yet'

valid runs: 45 of 47 completed


,pair,num_models,lr,features,distance,loss_con,loss_energy,runtime_s,iters_run,early_stopped,penalty_weight_final,con_ok,L_fade,under_fade,seed,config_key,error,gap_to_fade
26,frank25 -> clayton15,4,0.001,"[256, 256, 256]",0.031333,0.022330,0.000982,1758.892494,39999.0,True,5.0,True,0.024666,True,0,"{""features"": [256, 256, 256], ""lr"": 0.001, ""nu...",NaN,-0.006667
2,gaussian0.9 -> gaussian-0.9,4,0.001,"[256, 256, 256]",0.490791,0.013641,0.240876,1736.061171,39999.0,True,5.0,True,0.633184,True,0,"{""features"": [256, 256, 256], ""lr"": 0.001, ""nu...",NaN,0.142393
12,gaussian0.9 -> independence,4,0.001,"[128, 128, 128]",0.248836,0.005939,0.061920,3931.520175,109999.0,True,5.0,True,0.269088,True,0,"{""features"": [128, 128, 128], ""lr"": 0.001, ""nu...",NaN,0.020252
38,joe4 -> joe4@90,4,0.001,"[256, 256, 256]",0.432028,0.017218,0.186648,1798.774182,39999.0,True,5.0,True,0.498665,True,0,"{""features"": [256, 256, 256], ""lr"": 0.001, ""nu...",NaN,0.066636


In [9]:
# Marginal effect of each swept hyperparameter, averaged over the others.
# `features` arrives from the CSV as a JSON string (run_one serializes lists), which is
# exactly what groupby wants -- a list column would be unhashable.
# `n_diverged` counts against the full attempt table, not df_ok: a setting that wins on
# mean distance while diverging half the time has not won anything.
for h in SWEEP:
    print(f'--- {h} ---')
    stats = df_ok.groupby(h)[['distance', 'loss_con', 'runtime_s']].mean().round(4)
    stats['n'] = df_ok.groupby(h).size()
    stats['n_diverged'] = df.groupby(h).size() - stats['n']
    print(stats, '\n')

--- num_models ---
            distance  loss_con  runtime_s   n  n_diverged
num_models                                               
4             0.3073    0.0255  2445.6929  24           0
6             0.3115    0.0291  3291.6069  23           1 

--- lr ---
       distance  loss_con  runtime_s   n  n_diverged
lr                                                  
0.001    0.3053    0.0198  2042.2861  24           0
0.010    0.3136    0.0350  3712.5531  23           1 

--- features ---
                 distance  loss_con  runtime_s   n  n_diverged
features                                                      
[128, 128, 128]    0.3092    0.0278  3033.6892  16           0
[128, 128]         0.3080    0.0259  2276.2459  16           0
[256, 256, 256]    0.3110    0.0282  3296.3085  15           1 



## 7. Conclusion

**Chosen configuration: **`num_models=4`, `lr=1e-3`, `features=[256,256,256]`.

`lr=1e-3` beats `1e-2` on every measure, and `num_models=4` is slightly better and faster than `6`. The three network sizes do not make much of a difference. `[256,256,256]` was chosen because it gave the lowest valid distance on 3 of 4 pairs.